## Setup — compare LEGACY vs FIXED dataloader on x50 (t15 UOAI)

But: la xp `base_arctic_croscim_test_sit_UOAI_supervised_forecast_t15` donne de bonnes predictions avec le dataloader legacy (pre-juillet, `data_multires_supervised_legacy.py`) mais toujours mauvaises avec le pipeline gridref corrige (`data_multires_supervised.py`). Ce notebook instancie les deux datamodules avec la meme config (extraite du yaml de la xp) et compare, canal par canal, le meme patch x50 pour isoler ou la divergence restante se situe.

In [ ]:
%matplotlib inline

import os
os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'

import sys
sys.path.insert(0, '/Odyssey/private/m19beauc/4dvarnet-starter')

import numpy as np
import xarray as xr
import torch
import matplotlib.pyplot as plt
import cartopy
cartopy.config['pre_existing_data_dir'] = '/Odyssey/private/m19beauc/4dvarnet-starter/contrib/CROSCIM'
cartopy.config['data_dir'] = '/Odyssey/private/m19beauc/4dvarnet-starter/contrib/CROSCIM'
print('Setup OK')


## Config (mirrors `config/xp/CROSCIM/UNet_solvers/base_arctic_croscim_test_sit_UOAI_supervised_forecast_t15.yaml`)

In [ ]:
PATHS = dict(
    src        = '/Odyssey/private/m19beauc',
    mask       = '/Odyssey/private/m19beauc/4dvarnet-starter/contrib/CROSCIM/mask_PanArctic.nc',
    asip       = '/Odyssey/public/CROSCIM_dataset/ASIP_L3',
    cimr       = '/Odyssey/public/CROSCIM_dataset/data_noise',
    cristal    = '/Odyssey/public/CROSCIM_dataset/out_CRISTAL',
    covariates = '/Odyssey/public/CROSCIM_dataset/atm_data',
    models     = '/Odyssey/public/CROSCIM_dataset/out_MOD',
)

SATELLITE_VARS = {
    'cristal': ['SIT'],
    'cimr':    ['SIT', 'SIC'],
    'asip':    ['sic'],
}
MODELS_VARS = ['SIT']
COVARIATES  = ['t2m', 'msl', 'u10', 'v10']

TARGET_VARS = {
    'patch_x50': ['models_SIT'],
    'patch_x10': ['models_SIT'],
}
VAR_MAPPING = {
    'patch_x50': {'models_SIT': 'cristal_SIT'},
    'patch_x10': {'models_SIT': 'cristal_SIT'},
}

NORM_STATS = {
    'asip': {
        'sic': {'min': 0.0, 'max': 100.0, 'type': 'minmax'},
        'standard_deviation_sic': {'mean': 1.8761531586097209, 'std': 4.922335431544444, 'type': 'zscore'},
        'status_flag': {'min': 0.0, 'max': 1536.0, 'type': 'minmax'},
    },
    'cimr': {
        'SIC': {'min': 0.0, 'max': 1.0, 'type': 'minmax'},
        'SIT': {'mean': 0.5695980677516013, 'std': 0.8216149733058172, 'type': 'zscore'},
        'Tsurf': {'mean': -6.3703807274976665, 'std': 9.394544766238061, 'type': 'zscore'},
    },
    'cristal': {
        'HS': {'mean': 0.15517908473300512, 'std': 0.13962813089282153, 'type': 'zscore'},
        'SIT': {'mean': 0.5695980677516013, 'std': 0.8216149733058172, 'type': 'zscore'},
        'SSH': {'mean': 0.1912636630889516, 'std': 0.41697635927509086, 'type': 'zscore'},
    },
}
NORM_STATS_MODELS = {
    'SIC': {'min': 0.0, 'max': 1.0, 'type': 'minmax'},
    'SIT': {'mean': 0.5695980677516013, 'std': 0.8216149733058172, 'type': 'zscore'},
    'HS': {'mean': 0.06844576249551623, 'std': 0.1137041260225963, 'type': 'zscore'},
    'SSH': {'mean': 0.21988234269231605, 'std': 0.2757902493660287, 'type': 'zscore'},
}
NORM_STATS_COVS = {
    'msl': {'mean': 101261.65922657882, 'std': 1112.7187312302365, 'type': 'zscore'},
    't2m': {'mean': -2.5985455071070955, 'std': 13.613745769755358, 'type': 'zscore'},
    'u10': {'mean': 0.6010106010949151, 'std': 4.545559141871505, 'type': 'zscore'},
    'v10': {'mean': -0.2104646166032582, 'std': 4.707062023219409, 'type': 'zscore'},
    'tcc': {'min': 1.0896958883677144e-05, 'max': 0.024753304198384285, 'type': 'minmax'},
    'd2m': {'mean': 2.4095910147380064e-08, 'std': 4.900962447374831e-08, 'type': 'zscore'},
    'ssrd': {'mean': 95.72918023746189, 'std': 93.44316534428312, 'type': 'zscore'},
    'strd': {'mean': 265.25246499467744, 'std': 63.12553499850183, 'type': 'zscore'},
    'tp': {'mean': 2.4095910147380064e-08, 'std': 4.900962447374831e-08, 'type': 'zscore'},
}

XRDS_KW = dict(
    patch_dims        = {'time': 15, 'yc': 256, 'xc': 256},
    strides           = {'time': 1,  'yc': 28,  'xc': 28},
    strides_test      = {'time': 1,  'yc': 200, 'xc': 200},
    patch_dims_dict   = {
        50: {'time': 15, 'yc': 294, 'xc': 304},
        10: {'time': 15, 'yc': 256, 'xc': 256},
    },
    strides_test_dict = {
        50: {'time': 1, 'yc': 294, 'xc': 304},
        10: {'time': 1, 'yc': 200, 'xc': 200},
    },
    domain_limits = {
        'xc': slice(-3_849_750., 3_749_750.),
        'yc': slice( 2_473_750., -4_896_250.),
    },
)

DOMAINS = {
    'train': {'time': slice('2022-01-01', '2022-12-31')},
    'val':   {'time': [slice('2022-05-01', '2022-06-30'), slice('2022-07-01', '2022-12-31')]},
    'test':  {'time': slice('2022-02-01', '2022-02-15')},
}

MULTIRES = [50, 10]
print('Config OK')


## Instancier les deux datamodules (LEGACY vs FIXED)

In [ ]:
import copy
from contrib.CROSCIM.dataloaders.load_data import get_paths_for_source

def build_dm(BaseDataModuleMultiRes_cls):
    # xrds_kw gets mutated in place (BaseDataModuleMultiRes.__init__ does
    # xrds_kw.pop('patch_dims_dict', ...)) — deep-copy so the two calls
    # (fixed/legacy) each get the full, untouched config.
    return BaseDataModuleMultiRes_cls(
        asip_paths       = get_paths_for_source('asip',       PATHS),
        cimr_paths       = get_paths_for_source('cimr',       PATHS),
        cristal_paths    = get_paths_for_source('cristal',    PATHS),
        covariates_paths = get_paths_for_source('covariates', PATHS),
        models_paths     = get_paths_for_source('models',     PATHS),
        satellite_vars   = SATELLITE_VARS,
        models_vars      = MODELS_VARS,
        covariates       = COVARIATES,
        target_vars      = TARGET_VARS,
        var_mapping      = VAR_MAPPING,
        multires         = MULTIRES,
        mask_path        = PATHS['mask'],
        domain_name      = 'arctic_croscim',
        domains          = DOMAINS,
        xrds_kw          = copy.deepcopy(XRDS_KW),
        dl_kw            = {'batch_size': 1, 'num_workers': 0},
        res              = 500,
        pads             = [False, False, True],
        norm_stats       = NORM_STATS,
        norm_stats_models= NORM_STATS_MODELS,
        norm_stats_covs  = NORM_STATS_COVS,
    )


In [ ]:
from contrib.CROSCIM.dataloaders.data_multires_supervised import BaseDataModuleMultiRes as BaseDataModuleMultiRes_fixed

print('Building FIXED datamodule...')
dm_fixed = build_dm(BaseDataModuleMultiRes_fixed)
dm_fixed.setup('test')
print('FIXED test sizes:', {res: len(ds) for res, ds in dm_fixed.test_ds.datasets.items()})


In [ ]:
from contrib.CROSCIM.dataloaders.data_multires_supervised_legacy import BaseDataModuleMultiRes as BaseDataModuleMultiRes_legacy

print('Building LEGACY datamodule...')
dm_legacy = build_dm(BaseDataModuleMultiRes_legacy)
dm_legacy.setup('test')
print('LEGACY test sizes:', {res: len(ds) for res, ds in dm_legacy.test_ds.datasets.items()})


## Comparer le meme patch x50, canal par canal

Essaie plusieurs `idx` si le premier tombe sur une zone sans donnee (tout NaN).

In [ ]:
ds_fixed_x50  = dm_fixed.test_ds.datasets[50]
ds_legacy_x50 = dm_legacy.test_ds.datasets[50]

print('len fixed x50 :', len(ds_fixed_x50))
print('len legacy x50:', len(ds_legacy_x50))

idx = 0
item_fixed  = ds_fixed_x50[idx]
item_legacy = ds_legacy_x50[idx]

print('Fields (fixed): ', item_fixed._fields)


In [ ]:
def get_field(item, name):
    return np.asarray(getattr(item, name)) if hasattr(item, name) else None

KEYS = ['asip_sic', 'cimr_SIC', 'cimr_SIT', 'cristal_SIT', 'models_SIT', 'land_mask', 'xc', 'yc']

print(f"{'field':14s} {'shape_fixed':16s} {'shape_legacy':16s} {'max|diff|':>12s} {'mean|diff|':>12s} {'nan%_fixed':>11s} {'nan%_legacy':>12s}")
for key in KEYS:
    a = get_field(item_fixed, key)
    b = get_field(item_legacy, key)
    if a is None or b is None:
        print(f'{key:14s} MISSING (fixed={a is not None}, legacy={b is not None})')
        continue
    if a.shape != b.shape:
        print(f'{key:14s} SHAPE MISMATCH fixed={a.shape} legacy={b.shape}')
        continue
    diff = np.abs(a.astype(np.float64) - b.astype(np.float64))
    finite = np.isfinite(diff)
    maxdiff  = np.nanmax(diff) if finite.any() else float('nan')
    meandiff = np.nanmean(diff[finite]) if finite.any() else float('nan')
    nan_a = 100 * np.isnan(a).mean() if a.dtype.kind == 'f' else 0.0
    nan_b = 100 * np.isnan(b).mean() if b.dtype.kind == 'f' else 0.0
    print(f'{key:14s} {str(a.shape):16s} {str(b.shape):16s} {maxdiff:12.6g} {meandiff:12.6g} {nan_a:11.1f} {nan_b:12.1f}')


## Comparer TOUS les patchs x50 (pas juste idx=0)

idx=0 tombe sur un patch très NaN (bord de domaine) — peu représentatif.
On boucle sur tous les indices pour voir si la divergence apparaît ailleurs
(patch central, chevauchement...).

In [ ]:
KEYS = ['asip_sic', 'cimr_SIC', 'cimr_SIT', 'cristal_SIT', 'models_SIT', 'land_mask']

def compare_all_patches(res):
    ds_f = dm_fixed.test_ds.datasets[res]
    ds_l = dm_legacy.test_ds.datasets[res]
    n = len(ds_f)
    assert n == len(ds_l), f"x{res}: dataset length mismatch: fixed={n} legacy={len(ds_l)}"

    worst = {k: (0.0, None) for k in KEYS}
    n_diverging = {k: 0 for k in KEYS}

    for idx in range(n):
        item_f = ds_f[idx]
        item_l = ds_l[idx]
        for key in KEYS:
            a = get_field(item_f, key)
            b = get_field(item_l, key)
            if a is None or b is None or a.shape != b.shape:
                print(f'x{res} idx={idx} {key}: MISSING/SHAPE MISMATCH')
                continue
            diff = np.abs(a.astype(np.float64) - b.astype(np.float64))
            finite = np.isfinite(diff)
            maxdiff = np.nanmax(diff) if finite.any() else 0.0
            if maxdiff > 1e-6:
                n_diverging[key] += 1
            if maxdiff > worst[key][0]:
                worst[key] = (maxdiff, idx)

    print(f'\n=== x{res}: {n} patches compared ===')
    print(f"{'field':14s} {'#patches divergents':>20s} {'max|diff| (idx)':>20s}")
    for key in KEYS:
        maxdiff, idx = worst[key]
        print(f'{key:14s} {n_diverging[key]:20d} {maxdiff:14.6g} (idx={idx})')
    return n_diverging

n_div_50 = compare_all_patches(50)
n_div_10 = compare_all_patches(10)


## Visualiser les canaux qui divergent le plus (t=0)

Adapte la liste `KEYS_TO_PLOT` en fonction du tableau ci-dessus.

In [ ]:
KEYS_TO_PLOT = ['asip_sic', 'cimr_SIC', 'cristal_SIT']

fig, axes = plt.subplots(len(KEYS_TO_PLOT), 3, figsize=(15, 5 * len(KEYS_TO_PLOT)))
if len(KEYS_TO_PLOT) == 1:
    axes = axes[None, :]

for row, key in enumerate(KEYS_TO_PLOT):
    a = get_field(item_fixed, key)
    b = get_field(item_legacy, key)
    if a is None or b is None:
        continue
    a0, b0 = a[0], b[0]
    vmin = np.nanmin([np.nanmin(a0), np.nanmin(b0)])
    vmax = np.nanmax([np.nanmax(a0), np.nanmax(b0)])
    im0 = axes[row, 0].imshow(a0, vmin=vmin, vmax=vmax)
    axes[row, 0].set_title(f'{key} — FIXED (t0)')
    plt.colorbar(im0, ax=axes[row, 0], fraction=0.046)
    im1 = axes[row, 1].imshow(b0, vmin=vmin, vmax=vmax)
    axes[row, 1].set_title(f'{key} — LEGACY (t0)')
    plt.colorbar(im1, ax=axes[row, 1], fraction=0.046)
    d = a0.astype(np.float64) - b0.astype(np.float64)
    im2 = axes[row, 2].imshow(d, cmap='RdBu_r', vmin=-np.nanmax(np.abs(d)), vmax=np.nanmax(np.abs(d)))
    axes[row, 2].set_title(f'{key} — FIXED - LEGACY')
    plt.colorbar(im2, ax=axes[row, 2], fraction=0.046)

plt.tight_layout()
plt.show()
